# Use case: a semantic cache for expensive calls

An LLM call (or a slow API) is asked the same thing in different words all day: *"reset my
password"*, *"how do I change my password?"*, *"forgot password"*. An exact-match cache never
hits. A cache keyed by **meaning** does, but it can also serve the wrong answer to a question
that only *looks* the same. This notebook builds the cache on the low-level `Memory`, measures
where a cosine threshold works and where it cannot, and then adds the check that makes it safe.

**Needs:** Ollama with `nomic-embed-text` and `qwen2.5:7b-instruct`. The "expensive" call is simulated.

In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
STORE = ROOT / ".usecase_nb" / "cache"          # the store lives here; delete the folder to start over
RUN_LLM = True                               # the steps that call a chat model are slow on CPU
LLM = "qwen2.5:7b-instruct"                      # follows context better than llama3.2:3b

import time, hashlib
from slim_llm_memory import Embedder, Memory
from slim_llm_memory.llm import chat

mem = Memory(STORE, Embedder.ollama("nomic-embed-text"))

def expensive(question):
    """Stand-in for a long generation or a paid API: slow, deterministic so the demo is checkable."""
    time.sleep(1.0)
    return f"answer#{hashlib.sha1(question.encode()).hexdigest()[:6]} for: {question}"

In [2]:
class SemanticCache:
    """Nearest cached question by cosine; serve its answer when the score clears the threshold
    and, if a judge is given, when the judge agrees the two questions want the same answer."""
    def __init__(self, mem, threshold, judge=None):
        self.mem, self.threshold, self.judge = mem, threshold, judge
        self.hits = self.misses = 0

    def get(self, question):
        found = self.mem.search(question, k=1, kinds={"cache"}, min_score=self.threshold)
        if found and (self.judge is None or self.judge(question, found[0].text)):
            self.hits += 1
            return found[0].meta["answer"], found[0].text
        self.misses += 1
        answer = expensive(question)
        self.mem.upsert([{"id": hashlib.sha1(question.encode()).hexdigest(), "text": question,
                          "meta": {"kind": "cache", "answer": answer}}])
        return answer, None

## 1. Warm-up, then a read-only probe

Four distinct questions go in cold. Then eight rewordings and two **traps**: questions that
share most of their words with a cached one but need a different answer. For each, the nearest
cached question and its cosine, without touching the cache.

In [3]:
COLD = [
    "How do I reset my password?",
    "What is your refund policy?",
    "How long does standard shipping take?",
    "Can I change the email on my account?",
]
warm = SemanticCache(mem, threshold=1.01)          # threshold above 1: everything misses, everything is stored
for q in COLD:
    warm.get(q)
print("cached:", warm.misses, "questions")

PARA = [
    ("forgot my password, how do I get back in?",     COLD[0]),
    ("password reset link please",                    COLD[0]),
    ("will I get my money back if I return it?",      COLD[1]),
    ("refunds: how do they work?",                    COLD[1]),
    ("when will my order arrive with normal delivery?", COLD[2]),
    ("delivery time for regular shipping",            COLD[2]),
    ("update the email address on my profile",        COLD[3]),
    ("switch my login email to a new one",            COLD[3]),
]
TRAPS = [
    "How do I reset my two-factor authentication?",   # shares 'reset'; not the password question
    "How long does express shipping take?",           # one word away from the standard-shipping question
]

def nearest(q):
    top = mem.search(q, k=1, kinds={"cache"}, min_score=0.0)[0]
    return top.text, top.score

probe = {q: nearest(q) for q in [p for p, _ in PARA] + TRAPS}
print(f"\n{'question':<50} {'score':>5}  nearest cached question")
for q, want in PARA:
    text, score = probe[q]
    print(f"{q:<50} {score:5.2f}  {text}")
for q in TRAPS:
    text, score = probe[q]
    print(f"{q:<50} {score:5.2f}  {text}   ← TRAP: must not be served")

cached: 4 questions



question                                           score  nearest cached question
forgot my password, how do I get back in?           0.82  How do I reset my password?
password reset link please                          0.75  How do I reset my password?
will I get my money back if I return it?            0.62  What is your refund policy?
refunds: how do they work?                          0.77  What is your refund policy?
when will my order arrive with normal delivery?     0.69  How long does standard shipping take?
delivery time for regular shipping                  0.83  How long does standard shipping take?
update the email address on my profile              0.76  Can I change the email on my account?
switch my login email to a new one                  0.75  Can I change the email on my account?
How do I reset my two-factor authentication?        0.76  How do I reset my password?   ← TRAP: must not be served
How long does express shipping take?                0.83  How long does st

## 2. No cosine threshold separates them

Sweep the threshold. A rewording is *served* when its score clears it; a trap is *wrongly
served* when its score does too. The rows show the problem: the trap scores sit inside the
range of the honest rewordings.

In [4]:
for thr in [0.6, 0.7, 0.75, 0.8, 0.85]:
    good = sum(probe[q][1] >= thr and probe[q][0] == want for q, want in PARA)
    bad = sum(probe[q][1] >= thr for q in TRAPS)
    print(f"threshold {thr:.2f}: {good}/{len(PARA)} rewordings served · {bad}/{len(TRAPS)} traps wrongly served")

threshold 0.60: 8/8 rewordings served · 2/2 traps wrongly served
threshold 0.70: 6/8 rewordings served · 2/2 traps wrongly served
threshold 0.75: 5/8 rewordings served · 2/2 traps wrongly served
threshold 0.80: 2/8 rewordings served · 1/2 traps wrongly served
threshold 0.85: 0/8 rewordings served · 0/2 traps wrongly served


That is the embedder, not a bug: *"express shipping"* and *"standard shipping"* mean almost the
same thing to a bi-encoder, and a bi-encoder is what makes the cache fast. The same probe with
`bge-m3` and with the `bge-reranker-v2-m3` cross-encoder gave the same overlap.

## 3. Add a judge

A cheap second check: when the cosine shortlist finds a candidate, ask a small model one
question, *"would the same answer satisfy both?"*, with a three-token reply. It costs one short
LLM call, far less than the expensive call it protects, and it only runs when there is a
candidate. `qwen2.5:7b-instruct` got all ten pairs right here; `llama3.2:3b` got four wrong in the
same test, so measure your judge too.

In [5]:
JUDGE_SYSTEM = ("You compare two customer questions. Answer YES if the same answer would fully "
                "satisfy both, otherwise NO. Reply with one word.")

def same_question(a, b):
    out = chat(LLM, [{"role": "system", "content": JUDGE_SYSTEM},
                     {"role": "user", "content": f"A: {a}\nB: {b}"}], options={"num_predict": 3})
    return out.strip().upper().startswith("YES")

if RUN_LLM:
    print(f"{'verdict':<8} {'want':<5} question")
    for q, _ in PARA:
        print(f"{'YES' if same_question(q, probe[q][0]) else 'NO':<8} {'YES':<5} {q}")
    for q in TRAPS:
        print(f"{'YES' if same_question(q, probe[q][0]) else 'NO':<8} {'NO':<5} {q}")

verdict  want  question


YES      YES   forgot my password, how do I get back in?


YES      YES   password reset link please


YES      YES   will I get my money back if I return it?


YES      YES   refunds: how do they work?


...

YES      YES   delivery time for regular shipping


YES      YES   update the email address on my profile


YES      YES   switch my login email to a new one


NO       NO    How do I reset my two-factor authentication?


NO       NO    How long does express shipping take?


## 4. Live run

The judged cache with a permissive cosine shortlist (0.6, so every honest rewording reaches the
judge). Hits return in one embedding call plus one short judge call. The two traps and any
rewording the judge rejects fall through to `expensive()` and are stored, so the next similar
question hits.

Read the timings with the machine in mind: this run was on a CPU under heavy load, where the
embedding plus the judge call take around 20 seconds together, far more than the one-second
`expensive()` stand-in. On a GPU both are well under a second. The cache pays off when the call
it protects is a long generation or a rate-limited paid API, not a one-second function.

In [6]:
if RUN_LLM:
    cache = SemanticCache(mem, threshold=0.6, judge=same_question)
    for q in [p for p, _ in PARA] + TRAPS:
        t0 = time.perf_counter()
        answer, matched = cache.get(q)
        print(f"{'HIT ' if matched else 'MISS'} {(time.perf_counter() - t0) * 1000:6.0f} ms  {q}")
    print(f"\nhits {cache.hits} · misses {cache.misses} · entries stored: {mem.stats()['items_open']}")

HIT   28152 ms  forgot my password, how do I get back in?


HIT   19022 ms  password reset link please


HIT   26294 ms  will I get my money back if I return it?


HIT   24215 ms  refunds: how do they work?


HIT   20586 ms  when will my order arrive with normal delivery?


HIT   21185 ms  delivery time for regular shipping


HIT   17151 ms  update the email address on my profile


HIT   19061 ms  switch my login email to a new one


MISS  29492 ms  How do I reset my two-factor authentication?


MISS  23366 ms  How long does express shipping take?

hits 8 · misses 2 · entries stored: 6


## Takeaways

- **Question as text, answer in `meta`.** `Memory` needs nothing else; `kinds={"cache"}` keeps other items out of the way.
- **A cosine threshold alone is not safe.** Build a probe with rewordings *and* traps from your own traffic and sweep it; here no threshold served the rewordings without serving a trap.
- **Shortlist by cosine, confirm with a judge.** One short model call per candidate, none when there is no candidate. Worth it only when the protected call costs more than the judge does; measure both on your hardware.
- **Eviction:** `mem.remove(id)`; store a timestamp in `meta` and sweep old entries.

In [7]:
mem.close()